# ARO 3D Studio — engine lab (Colab)

**What this does:** runs *stronger* image-to-3D engines than TripoSR on your real reference images (a truck and a car), through the Studio's own pipeline, and puts the results next to the TripoSR results we already measured for the same images. For every engine and image you get: the raw GLB exactly as the engine returned it, the prepared game-ready GLB (LODs, collision, validation), a Unity package, and a **contact sheet** (reference, outline overlay, front, side, rear, top, three-quarter) plus measured numbers (silhouette overlap, length/width/height proportions).

**Engines (free T4 friendly):** TripoSG (geometry, MIT code) and, only if you switch it on, Hunyuan3D-2 mini (geometry, *restricted licence, evaluation only*). Geometry-only engines return grey meshes: the Studio **projects your reference photo onto them** for colour (the unseen side is inferred and reported as such).

**What it does not prove:** that a model looks good (you must look at the sheets), that it is production-ready, or anything about Unity 6 (import the Unity package and follow `docs/unity/` for that). TRELLIS.2 (the best open textured engine) needs a 24 GB-class GPU and is not run here.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run the cells **in order**. No secret is needed. Total time on a free T4: roughly 15-25 min for the one-time setup (it downloads PyTorch and the model weights, several GB), then a few minutes per image per engine.

If anything fails: run the last cell (failure bundle) and send me `aro-lab-logs.zip` plus a screenshot of the failing cell.

## 1 · GPU check
**Expected:** a line with `Tesla T4` (about 15 GB) or a bigger card. If it prints nothing, this is not a GPU runtime: stop and change the runtime type.

In [ ]:
import os, shutil
os.makedirs('/content/lab', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print('free disk GB:', round(shutil.disk_usage('/content').free / 1e9, 1), '(need about 25)')
assert shutil.which('nvidia-smi'), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'

## 2 · Get the Studio
**Expected:** the last line prints the commit you are testing. Keep it: it goes into the evidence.

In [ ]:
STUDIO_REF = 'claude/zen-johnson-je3dzk'   # branch or full commit hash
if not os.path.isdir('/content/aro'):
    !git clone -q https://github.com/AffanNamu/ARO-3D-STUDIO /content/aro
%cd /content/aro
!git fetch -q origin
is_branch = os.system(f'git show-ref --verify --quiet refs/remotes/origin/{STUDIO_REF}') == 0
if is_branch:
    !git checkout -q -B $STUDIO_REF origin/$STUDIO_REF
else:
    !git checkout -q $STUDIO_REF
!git log -1 --format='STUDIO COMMIT %H  %s'
!pip install -q -r requirements.txt > /content/lab/pip-studio.log 2>&1 || (tail -40 /content/lab/pip-studio.log; false)
print('studio requirements installed')

## 3 · Build the GPU worker environment (one time, about 10-20 minutes)
The engines need Python 3.10 and an older PyTorch than this notebook runtime ships, so they get their **own isolated environment** (no root, nothing changed in the notebook's Python). The Studio then talks to the worker over localhost HTTP, which is exactly how it will talk to a rented GPU server later.

**Expected:** progress lines (`creating an isolated Python 3.10 environment`, `installing PyTorch`, `installing triposg`), then a JSON block whose `engines_ready` lists the engines and whose `venv_python_torch_cuda` ends with `True`. The full log is `/content/lab/setup.log`; on failure the last lines are printed.

**Hunyuan3D-2 mini is evaluation only** (its licence excludes the EU, UK and South Korea and forbids training other models on its output). Leave `EVALUATE_RESTRICTED = False` unless you have read `docs/models/model-licenses.md` section 2b and accept that this run is only a quality yardstick.

In [ ]:
EVALUATE_RESTRICTED = False     # True also sets up Hunyuan3D-2 mini (restricted licence, evaluation only)
ENGINES = 'triposg,hunyuan3d' if EVALUATE_RESTRICTED else 'triposg'
!cd /content/aro && python scripts/lab_bootstrap.py setup --engines $ENGINES --root /content/lab

## 3b · Optional: Blender for independent checks (about 3-5 minutes)
Installs Blender as a Python module in its own environment. With it, every result also gets **beauty renders made by Blender** (real texture sampling, normal map, lighting: an independent reader of the exported GLB, not our own viewer) and an **FBX export that Blender re-imports and checks** (triangles, size in metres, materials, maps). Set `WITH_BLENDER = False` to skip; nothing else depends on it.


In [ ]:
WITH_BLENDER = True
if WITH_BLENDER:
    import sys
    sys.path.insert(0, '/content/aro/scripts')
    import lab_bootstrap as lb
    from pathlib import Path
    blender_python = lb.setup_blender(Path('/content/lab'))
    if blender_python:
        os.environ['ARO_BLENDER_PYTHON'] = blender_python   # the bake-off below picks it up
else:
    print('skipped')

## 4 · Start the worker
**Expected:** a table with one row per engine. `available: True` means the worker can run it. `False` shows the exact reason (for example a missing package): send me that table. The weights are downloaded on the first real run, not now.

In [ ]:
import sys, json, urllib.request
sys.path.insert(0, '/content/aro/scripts')
import lab_bootstrap as lb
from pathlib import Path
info = lb.start(ENGINES.split(','), Path('/content/lab'), 8000, None)
os.environ['ARO_WORKER_TOKEN'] = info['token']          # kept in the environment only, never printed or written to a file
WORKER_URL = info['url']
req = urllib.request.Request(WORKER_URL + '/v1/engines', headers={'Authorization': 'Bearer ' + info['token']})
man = json.load(urllib.request.urlopen(req))
print('GPU:', man['gpu'])
for e in man['engines']:
    print(f"{e['id']:10s} available: {e['available']}  version: {e.get('version')}")
    for r in e.get('reasons', []): print('     reason:', r)
    for w in e.get('warnings', []): print('     note  :', w)

## 5 · Your reference images
Upload one truck and one car image (the same ones we measured with TripoSR if you can: for example `ARO 480 Truck at Golden Hour.png` and `Premium Blue SUV Studio Render.png`; the results are then compared with those TripoSR numbers automatically). Use a clear, roughly three-quarter view of the whole vehicle.

`CATEGORY` and `LENGTH_M` are per file: a wrong category only changes guidance text. Enter a real length in metres only if you **know** it (otherwise leave `None`: the model keeps a 1 m placeholder and says so).

In [ ]:
from google.colab import files
os.makedirs('/content/lab/inputs', exist_ok=True)
up = files.upload()
CASES = []
for name, data in up.items():
    path = '/content/lab/inputs/' + os.path.basename(name)
    open(path, 'wb').write(data)
    low = name.lower()
    category = 'heavy_truck' if any(k in low for k in ('truck', 'hauler', 'lorry', 'tractor', '480')) else 'car'
    CASES.append((path, category, None))   # edit here if the guess is wrong or you know the real length: (path, 'car', 4.5)
for c in CASES: print(c)

## 6 · Run the engines
**Expected:** for each image and engine, progress lines; the first run of an engine also downloads its weights (a few GB). When it finishes a table is printed: TripoSR's earlier result for the same image first, then each engine.

`I_HAVE_READ_THE_MODEL_CARDS` is **your licence attestation**, the same as the `ARO_VERIFIED_MODEL_LICENSES` setting: leave it `False` and the Studio refuses to run the engines. Set it to `True` only after reading the TripoSG model card (`huggingface.co/VAST-AI/TripoSG`) and, if you enabled it, the Hunyuan3D terms.

`PRESET` is `STANDARD` (50 steps, 512-grid) by default; use `DRAFT` for a quick first look.

In [ ]:
I_HAVE_READ_THE_MODEL_CARDS = False
PRESET = 'STANDARD'
OUT = '/content/lab/results'
case_args = ' '.join("--case '%s:%s%s'" % (p, c, (':%s' % l) if l else '') for p, c, l in CASES)
verified = ENGINES if I_HAVE_READ_THE_MODEL_CARDS else ''
!cd /content/aro && python scripts/engine_bakeoff.py --worker-url $WORKER_URL --engines $ENGINES $case_args --preset $PRESET --verified-licences "$verified" --out $OUT

## 7 · LOOK at the results (this is the quality check)
Each sheet: reference, outline overlay (yellow = agreement, red = reference only, green = model only), best-matching view, front, side, rear, top, three-quarter. Judge with your eyes: proportions, cab and body shape, wheels as separate round shapes, surface smoothness, colour where the photo shows it. The numbers only measure outline and proportions, never detail or taste.

In [ ]:
import lab_notebook as ln
ln.show_results(Path(OUT))

## 8 · Package the evidence
Two zips are downloaded: `lab-evidence-repo.zip` (numbers, sheets, prepared GLBs, Unity packages, environment, logs: safe to commit) and `lab-raw-meshes.zip` (the engines' untouched GLBs: large, keep them, do not commit). Send me `lab-evidence-repo.zip`.

In [ ]:
zips = ln.package(Path(OUT), Path('/content/lab'), Path('/content/lab/package'))
for z in zips: print(z, round(z.stat().st_size / 1e6, 1), 'MB')
for z in zips: files.download(str(z))

## 9 · Unity 6 (do this on your machine with the editor)
Each run folder has `unity-package.zip`. Import the best one into a Unity 6 project and follow `docs/verification/unity-verification-runbook.md` (it ships a batch-mode verifier). Nothing in this notebook has touched Unity: until you run that, the assets are **not** verified in Unity.

When you are done, stop the worker (frees the GPU).

In [ ]:
lb.stop(Path('/content/lab'))

## 10 · Failure bundle (run this if ANY cell failed)
Bundles the logs and the worker log (no weights, no tokens).

In [ ]:
!cd /content && zip -qr aro-lab-logs.zip lab/setup.log lab/setup.json lab/worker.log lab/pip-studio.log lab/results/bakeoff-summary.json lab/results/bakeoff-summary.md 2>/dev/null; ls -l aro-lab-logs.zip
from google.colab import files
files.download('/content/aro-lab-logs.zip')